# Notebook 12 - Phase Analysis

## Goal
Inspect phase and group-delay-like behavior for deepfake artifacts.


## Agenda
- Compute STFT
- Extract phase
- Unwrap and differentiate
- Visualize group-delay-like map


## Concept and Math

Phase = angle of complex spectrum.
Group delay is linked to frequency derivative of phase.
Phase inconsistencies can reveal synthesis artifacts missed by magnitude-only features.


In [ ]:
from pathlib import Path
import numpy as np
import librosa as lb
import librosa.display
import matplotlib.pyplot as plt

DATA_ROOT = Path("../dataset")
audio_files = sorted(DATA_ROOT.rglob("*.flac")) + sorted(DATA_ROOT.rglob("*.wav"))
if not audio_files:
    raise FileNotFoundError("No .flac or .wav found under ../dataset")

audio_path = audio_files[0]
print(f"Using: {audio_path}")

wave, sr = lb.load(audio_path, sr=16000, mono=True)
S = lb.stft(wave, n_fft=512, hop_length=160, win_length=400)
phase = np.angle(S)
unwrapped = np.unwrap(phase, axis=0)
gd_like = -np.diff(unwrapped, axis=0)

print("phase:", phase.shape, "gd_like:", gd_like.shape)

plt.figure(figsize=(10, 4))
librosa.display.specshow(gd_like, sr=sr, hop_length=160, x_axis="time", y_axis="linear", cmap="coolwarm")
plt.colorbar()
plt.title("Group-delay-like map")
plt.tight_layout()
plt.show()


## PyTorch Equivalent Snippet
Understand the librosa block first, then map it to this snippet.


In [ ]:
import torch
import torchaudio

wave_t, sr_t = torchaudio.load(str(audio_path))
wave_t = wave_t.mean(dim=0)
S_t = torch.stft(wave_t, n_fft=512, hop_length=160, win_length=400, return_complex=True)
phase_t = torch.angle(S_t)
gd_like_t = -torch.diff(phase_t, dim=0)
print(phase_t.shape, gd_like_t.shape)


## Review Checklist
- Why can phase be more fragile in generated audio?
- How does unwrapping help analysis?
- What caution is needed for interpreting group-delay proxies?
